## Station Status Exploration Notebook

**Objective:** Which of the information now available can be used to predict either bike stockouts or
full stations at some future future time interval. This notebook uses development data, which is restricted
to a finite time range defined in the '00-build-development-data' notebook.

Four figure sheets cover temporal stockout patterns, geographic patterns, associations of different variables
with stockouts, and diagnostics of some basic models.

In [0]:
status = spark.table("citibike_project.citibike.dev_station_status")
station_info_raw = spark.table("citibike_project.citibike.dev_station_info")


In [0]:
from pyspark.sql import functions as F, Window
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable

import textwrap
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression,  RandomForestClassifier
from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorSlicer, Interaction, Imputer, VectorAssembler
)
from time import perf_counter



spark.conf.set("spark.sql.session.timeZone", "UTC")

status = spark.table("citibike_project.citibike.dev_station_status")
station_info = spark.table("citibike_project.citibike.dev_station_info")

LOCAL_TZ = "America/New_York"
HORIZON_MIN = 30 # This is not the world's most ambitious choice
ENDPOINT_TOLERANCE_MIN = 5
MAX_GAP_MIN = 10

MAX_OBSERVATION_GAP_MIN = 10
VALIDATION_START = "2026-09-21T04:00:00Z"  # We will have a separate validation set


# Helper function for graphing:

# Light backdrop for station maps: every known station location in pale gray,
# with the aspect ratio corrected so New York isn't stretched east-west.
background_stations = (
    station_info
    .filter("lat IS NOT NULL AND lon IS NOT NULL")
    .select("lat", "lon").distinct()
    .toPandas()
)

def map_background(ax):
    ax.set_facecolor("#f4f4f4")
    ax.scatter(background_stations.lon, background_stations.lat,
               s=3, color="#d0d0d0", zorder=1, linewidths=0)
    ax.set_aspect(1 / np.cos(np.radians(40.73)))

# One coverage rule:
# No observation gap may exceed 10 minutes, and coverage must reach
# the horizon with an eligible poll no later than 10 minutes afterward.
# Observations AFTER the horizon never create positive labels.

required_status = [
    "station_id", "fetched_at", "station_reported_at",
    "num_bikes_available", "num_bikes_disabled",
    "num_docks_available", "num_docks_disabled",
    "is_installed", "is_renting", "is_returning",
]

required_info = [
    "station_id", "fetched_at", "name", "capacity", "lat", "lon",
]


for name, frame, required in [
    ("status", status, required_status),
    ("station_info", station_info, required_info),
]:
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

status = status.select(*required_status)
station_info = station_info.select(*required_info).distinct()

bounds = status.agg(
    F.min("fetched_at").alias("first_poll"),
    F.max("fetched_at").alias("last_poll"),
).first()

split_time = (
    spark.range(1)
    .select(F.to_timestamp(F.lit(VALIDATION_START)).alias("time"))
    .first().time
)

if bounds.first_poll is None:
    raise ValueError("The saved development status table is empty.")
if not bounds.first_poll < split_time < bounds.last_poll:
    raise ValueError("VALIDATION_START must fall inside the saved development period.")

print("Development observations:", bounds.first_poll, "to", bounds.last_poll)
print("Chronological development validation begins:", split_time)


## 1. Data Preparation

1. The purpose of the next slide is to make sure there is no duplicate data
2. Join with station_info to add contextual/temporally appropriate information about the stations
3. Create the target variables, which we refer to as 'bike_stockout' or 'dock_stockout'
4. 


In [0]:
# First we check for duplicates or missing infomation. Have never yet had a problem here 

for name, frame in [("status", status), ("information", station_info)]:
    if frame.filter("station_id IS NULL OR fetched_at IS NULL").take(1): # Do we have timestamp and station id?
        # take(1) returns a list of up to 1 row
        raise ValueError(f"{name}: missing station or timestamp")

    duplicates = (
        frame.groupBy("station_id", "fetched_at") 
        .count().filter("count > 1") # Do we have more than one entry for a given station and timestamp?
    )
    if duplicates.take(1):
        display(duplicates.limit(10))
        raise ValueError(f"{name}: duplicate keys require investigation") # Error if we have this problem

# We are going to be work with the data partitioned by station_Id
# This ensures that neighborhing information in the data frame
# comes from the same station and enables easy use of window functions
# This could be a candidate for how we organize our overall data frames
# So as to save some computation 


info_order = Window.partitionBy("station_id").orderBy("fetched_at")
historical_info = (
    station_info
    .withColumn("next_info_at", F.lead("fetched_at").over(info_order))
    .withColumnRenamed("fetched_at", "info_at") # Both station_info and status have have fetched_at columns so rename one prior to join
)

# This is a an "as_of" left join between the station status and the station info.
# It adds the newest station info record that happened before the station status 
# time stamp

status, info = status.alias("status"), historical_info.alias("info")

observations = (
    status.join(
        info,
        (F.col("status.station_id") == F.col("info.station_id"))
        # Status time is at or after this information snapshot.
        & (F.col("status.fetched_at") >= F.col("info.info_at"))
        # Status time is before the next snapshot, if one exists.
        & (
            F.col("info.next_info_at").isNull()
            | (F.col("status.fetched_at") < F.col("info.next_info_at"))
        ),
        "left",
    ) # Keep all the status information and some of the station info columns
    .select(
        "status.*",
        "info.info_at",
        "info.capacity",
        "info.lat",
        "info.lon",
        F.col("info.name").alias("station_name"),
    )
)


# Now we construct our stockout columns. Have null columns if the station
# isn't renting or returining

for outcome, operation in [("bike", "is_renting"), ("dock", "is_returning")]:
    count = F.col(f"num_{outcome}s_available")
    eligible = (
        F.col("is_installed").cast("boolean")
        & F.col(operation).cast("boolean")
        & count.isNotNull()
        & (count >= 0)
    )
    observations = observations.withColumn(
        f"{outcome}_stockout",
        F.when(eligible, count == 0),
    )


# We are going to use these to calculate overall station capacity 

component_columns = [
    "num_bikes_available", "num_bikes_disabled",
    "num_docks_available", "num_docks_disabled",
]

# observations is the data frame that we are going to use for analysis.
# We add some basic features to it, like the "reported total"
# There will be some times in the data where the reported total is inconsistent
# with the official station capacity. The causes of this could be manifold, 
# I have identified station valet service and other moving operations as causes
# Regretably, despite GBFS 2.3 having a "is_valet" column citibike seems not to
# transmit it


observations = (
    observations
    .withColumn(
        "reported_total",
        sum(F.col(c).cast("double") for c in component_columns),
    )
    .withColumn("capacity_discrepancy", F.col("reported_total") - F.col("capacity")) # This measures capacity inconsistencies
    .withColumn("local_time", F.from_utc_timestamp("fetched_at", LOCAL_TZ)) # Local time
    .withColumn("hour", F.hour("local_time")) # Hour
    .withColumn("weekend", F.dayofweek("local_time").isin(1, 7).cast("double")) # Day of week
    .withColumn("poll_microseconds", F.expr("unix_micros(fetched_at)")) 
)

# This gives a quick summary of our data

display(observations.agg(
    F.count("*").alias("polls"),
    F.countDistinct("station_id").alias("stations"),
    F.sum(F.col("info_at").isNull().cast("long")).alias("missing_historical_info"),
    F.count("bike_stockout").alias("eligible_bike_polls"),
    F.count("dock_stockout").alias("eligible_dock_polls"),
    F.avg(F.col("bike_stockout").cast("double")).alias("bike_stockout_share"),
    F.avg(F.col("dock_stockout").cast("double")).alias("dock_stockout_share"),
))

## Capacity Inconsistencies

As mentioned in the previous slide, the total number of bikes and docks at a station can sometimes exceed that stations official capacity. I have 
strong suspicions about what some of these events are caused by, for example valet service where a person arrives at a very busy station with a huge truck either full or bikes or which can be filled with bikes, to temporarily enhance station capacity. Citibike notably mentions a few stations which regularly employ this service. Unfortunately, I have yet to find a flag which proves it is what happens. This is worth further investigation

In [0]:

# How often does the reported total exceed historical capacity?
capacity_check = observations.filter(
    "capacity > 0 AND reported_total IS NOT NULL"
)

display(capacity_check.agg(
    F.count("*").alias("comparable_polls"),
    (100 * F.avg(
        (F.col("capacity_discrepancy") > 0).cast("double")
    )).alias("percent_over_capacity"),
    F.max("capacity_discrepancy").alias("largest_excess"),
))

# Inspect the largest discrepancies and their component counts.
display(
    capacity_check
    .filter("capacity_discrepancy > 0")
    .select(
    "station_id", "station_name", "fetched_at", "capacity",
    *component_columns, "reported_total", "capacity_discrepancy",
)
    .orderBy(F.desc("capacity_discrepancy"))
    .limit(10)
)

## 2. Temporal and geographic patterns
Two figures describe current shortages, not future onset. Rates use separate eligibility denominators.
Geographic differences compare the same historical station locations; they are not matched day-by-day.


## Temporal Patterns:

1. No bikes is rarer than no docks by a factor of 3
2. No bikes peaks just after the morning commute on weekdays and only gets resolved overnight
3. No docks builds in the afternoon and peaks in the evening. Again it is resolved early in the morning, with a small secondary peak just past the morning commute
4. Weekend patterns are shifted later in the day


In [0]:
hourly = (
    observations.groupBy("hour", "weekend")
    .agg(*[
        F.avg(F.col(f"{k}_stockout").cast("double")).alias(k)
        for k in ["bike", "dock"]
    ])
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
for ax, outcome in zip(axes, ["bike", "dock"]):
    for weekend, name in [(0, "Weekday"), (1, "Weekend")]:
        q = hourly[hourly.weekend == weekend].sort_values("hour")
        ax.plot(q.hour, 100*q[outcome], label=name)
    ax.set(
        title=f"No {outcome}s",
        xlabel="New York hour",
        ylabel="Eligible polls (%)",
    )
    ax.legend()
plt.show()

In [0]:


geography = (
    observations
    .filter("lat IS NOT NULL AND lon IS NOT NULL")
    .withColumn("period", F.floor(F.col("hour") / 6))
    .groupBy("station_id", "lat", "lon", "period")
    .agg(
        F.count("bike_stockout").alias("bike_polls"),
        F.avg(F.col("bike_stockout").cast("double")).alias("bike_rate"),
        F.count("dock_stockout").alias("dock_polls"),
        F.avg(F.col("dock_stockout").cast("double")).alias("dock_rate"),
    )
    .toPandas()
)

period_names = [
    "Midnight–6am",
    "6am–noon",
    "Noon–6pm",
    "6pm–midnight",
]

fig, axes = plt.subplots(
    2, 4,
    figsize=(22, 15),
    constrained_layout=True,
)

rate_scale = Normalize(0, 50)




for row, outcome in enumerate(["bike", "dock"]):
    for period, period_name in enumerate(period_names):
        ax = axes[row, period]
        map_background(ax)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticks([])
        ax.set_yticks([])

        q = geography[
            (geography.period == period)
        ].sort_values(f"{outcome}_rate")

        ax.scatter(
            q.lon, q.lat,
            c=100 * q[f"{outcome}_rate"],
            cmap="magma_r", norm=rate_scale,
            s=8, zorder=2,
        )
        ax.set_title(
            f"No {outcome}s · {period_name}",    
            fontsize=18,
            pad=10
        )


colorbar = fig.colorbar(
    ScalarMappable(norm=rate_scale, cmap="magma_r"),
    ax=axes.ravel().tolist(),
    shrink=.6,
    extend="max",
)
colorbar.set_label("Eligible polls with a shortage (%)", fontsize=16)
colorbar.ax.tick_params(labelsize=14)

fig.suptitle(
    "Geographic shortage prevalence by local time",
    fontsize=26,
    fontweight="bold",
    x=0.05, y=0.99,
    ha="left",
)

fig.text(
    0.05, 0.95,
    "There are distinct patterns of outages. There is a persistent shortage of bikes in\n"
    "Midtown, Harlem/Upper Manhattan, the Bronx, and Brooklyn around Prospect Park. This shortage is only\n"
    "alleviated in Midtown after the morning commute. There is an extreme shortage of dock space\n"
    "near hospitals on the Upper East Side, in Williamsburg and Greenpoint, and along the\n"
    "peripheral boundary of the network in Brooklyn and Queens.",
    fontsize=14,
    ha="left",
    va="top",
    linespacing=1.4,
)

# Reserve space above the panels for the title and subtitle.
fig.get_layout_engine().set(rect=(0, 0, 1, 0.80))

plt.show()

In [0]:

fullness_geography = (
    observations
    .filter(
        "lat IS NOT NULL AND lon IS NOT NULL "
        "AND capacity > 0 AND bike_stockout IS NOT NULL"
    )
    .withColumn("period", F.floor(F.col("hour") / 6))
    .withColumn(
        "fullness",
        F.least(
            F.col("num_bikes_available") / F.col("capacity"),
            F.lit(1.0),
        ),
    )
    .groupBy("station_id", "lat", "lon", "period")
    .agg(F.avg("fullness").alias("mean_fullness"))
    .toPandas()
)

fig, axes = plt.subplots(
    1, 4,
    figsize=(22, 9),
    constrained_layout=True,
)

fullness_scale = Normalize(0, 1)

for ax, period, period_name in zip(axes, range(4), period_names):
    map_background(ax)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])

    q = fullness_geography[
        fullness_geography.period == period
    ].sort_values("mean_fullness")

    ax.scatter(
        q.lon, q.lat,
        c=q.mean_fullness,
        cmap="viridis", norm=fullness_scale,
        s=8, zorder=2,
    )
    ax.set_title(
        period_name,
        fontsize=18,
        pad=10,
    )

colorbar = fig.colorbar(
    ScalarMappable(norm=fullness_scale, cmap="viridis"),
    ax=axes.tolist(),
    shrink=.6,
)
colorbar.set_label("Mean available bikes / capacity", fontsize=16)
colorbar.ax.tick_params(labelsize=14)

fig.suptitle(
    "Mean station fullness by local time",
    fontsize=26,
    fontweight="bold",
    x=0.05, y=0.99,
    ha="left",
)

fig.text(
    0.05, 0.93,
    textwrap.fill(
        "Overall capacity, just like stockouts, exhibits substantial geographic heterogeneity. "
        "Geograpraphic influence persists in all time periods, some areas are permanently short bikes and others docks.",
        width=140,
    ),
    fontsize=16,
    ha="left",
    va="top",
    linespacing=1.4,
)

fig.get_layout_engine().set(rect=(0, 0, 1, 0.80))

plt.show()



## Research Gold Data Frame

Now we are going to create a data frame which emulates what would be in a gold table that we would use to directly fit models.

We will do this in two stages: (1) creating predictors and (2) creating the targets

### Predictors

We create the following potentially interesting and useful geographic features:


| Predictor | Meaning |
|---|---|
| `bike_available_fraction` | Available bikes / capacity |
| `dock_available_fraction` | Available docks / capacity |
| `bike_change_per_minute` | Bike change since the previous poll, per minute |
| `dock_change_per_minute` | Dock change since the previous poll, per minute |
| `bike_change_30min_per_minute` | Net bike change over approximately 30 minutes, per minute |
| `dock_change_30min_per_minute` | Net dock change over approximately 30 minutes, per minute |
| `bike_change_per_minute_fraction` | Latest bike change per minute ÷ capacity |
| `dock_change_per_minute_fraction` | Latest dock change per minute ÷ capacity |
| `bike_change_30min_per_minute_fraction` | Longer-window bike change per minute ÷ capacity |
| `dock_change_30min_per_minute_fraction` | Longer-window dock change per minute ÷ capacity |
| `hour_sin_1`, `hour_cos_1` | 24-hour cycle |
| `hour_sin_2`, `hour_cos_2` | 12-hour cycle |
| `log_capacity` | Log of  station capacity |
| `geographic_cell` | Approximate kilometre-scale location category. x and y grid coordinates, very crude |




In [0]:
# Predictors available at the current observation time.


MICROSECONDS_PER_MINUTE = 60000000

station_order = Window.partitionBy("station_id").orderBy("poll_microseconds")
recent_window = station_order.rangeBetween(
    -30 * MICROSECONDS_PER_MINUTE, 0
)

features = observations.withColumn(
    "gap_minutes",
    (
        F.col("poll_microseconds")
        - F.lag("poll_microseconds").over(station_order)
    ) / MICROSECONDS_PER_MINUTE,
)

# Daily harmonics: 24-hour and 12-hour cycles.
fractional_hour = (
    F.col("hour")
    + F.minute("local_time") / 60
    + F.second("local_time") / 3600
)

harmonic_features = []

for frequency in [1, 2]:
    for name, function in [("sin", F.sin), ("cos", F.cos)]:
        column = f"hour_{name}_{frequency}"
        features = features.withColumn(
            column,
            function(2 * np.pi * frequency * fractional_hour / 24),
        )
        harmonic_features.append(column)

# Station size and approximate kilometre-scale geographic grouping.
features = (
    features
    .withColumn(
        "log_capacity",
        F.when(F.col("capacity") > 0, F.log("capacity")),
    )
    .withColumn(
        "geographic_cell",
        F.when(
            F.col("lat").isNotNull() & F.col("lon").isNotNull(),
            F.concat_ws(
                ":",
                F.floor((F.col("lon") + 74) * 84.3).cast("string"),
                F.floor((F.col("lat") - 40.7) * 111.2).cast("string"),
            ),
        ).otherwise("unknown"),
    )
)

for outcome in ["bike", "dock"]:
    state = f"{outcome}_stockout"

    available = F.when(
        F.col(state).isNotNull(),
        F.col(f"num_{outcome}s_available").cast("double"),
    )

    # Most recent change, in bikes or docks per minute.
    features = features.withColumn(
        f"{outcome}_change_per_minute",
        F.when(
            (F.col("gap_minutes") > 0)
            & (F.col("gap_minutes") <= MAX_OBSERVATION_GAP_MIN),
            (
                available - F.lag(available).over(station_order)
            ) / F.col("gap_minutes"),
        ),
    )

    # Earliest observation within the preceding 30 minutes.
    first_observation = F.first(
        F.struct(
            F.col("poll_microseconds").alias("time"),
            available.alias("available"),
        )
    ).over(recent_window)

    elapsed_minutes = (
        F.col("poll_microseconds") - first_observation["time"]
    ) / MICROSECONDS_PER_MINUTE

    # Exclude the gap leading into the first row of this history window.
    internal_gaps = (
        F.count("*").over(recent_window) - 1
        == F.sum(
            F.when(
                F.col("gap_minutes").between(0.000001, MAX_OBSERVATION_GAP_MIN),
                1,
            ).otherwise(0)
        ).over(recent_window)
        - F.first(
            F.when(
                F.col("gap_minutes").between(0.000001, MAX_OBSERVATION_GAP_MIN),
                1,
            ).otherwise(0)
        ).over(recent_window)
    )

    eligible_history = (
        F.count(state).over(recent_window)
        == F.count("*").over(recent_window)
    )

    features = features.withColumn(
        f"{outcome}_change_30min_per_minute",
        F.when(
            (elapsed_minutes >= 20) & internal_gaps & eligible_history,
            (available - first_observation["available"]) / elapsed_minutes,
        ),
    )

    # Normalize by positive historical capacity.
    capacity = F.when(F.col("capacity") > 0, F.col("capacity"))

    features = features.withColumn(
        f"{outcome}_available_fraction",
        available / capacity,
    )

    for change in ["change_per_minute", "change_30min_per_minute"]:
        features = features.withColumn(
            f"{outcome}_{change}_fraction",
            F.col(f"{outcome}_{change}") / capacity,
        )

## Target Construction

There are two target variables, 'bike_target' and 'dock_target'. Null means either some data is missing or the station is already in a stockout for bikes or docks. The target variables are ahead of time of the predictor variabels according to whatever time horizon we selected at the beginning.

In [0]:

horizon = HORIZON_MIN * MICROSECONDS_PER_MINUTE
maximum_gap = MAX_OBSERVATION_GAP_MIN * MICROSECONDS_PER_MINUTE

future_window = station_order.rangeBetween(1, horizon)
endpoint_window = station_order.rangeBetween(
    horizon, horizon + maximum_gap
)

features = (
    features
    .withColumn("future_poll_count", F.count("*").over(future_window))
    .withColumn(
        "last_future_poll",
        F.max("poll_microseconds").over(future_window),
    )
    .withColumn(
        "largest_future_gap",
        F.max("gap_minutes").over(future_window),
    )
    .withColumn(
        "endpoint",
        F.min(
            F.struct("poll_microseconds", "bike_stockout", "dock_stockout")
        ).over(endpoint_window),
    )
    .withColumn(
        "label_end_microseconds",
        F.col("endpoint.poll_microseconds"),
    )
)

for outcome in ["bike", "dock"]:
    state = f"{outcome}_stockout"

    covered = (
        (F.col("future_poll_count") > 0)
        & (
            F.count(state).over(future_window)
            == F.col("future_poll_count")
        )
        & (F.col("largest_future_gap") <= MAX_OBSERVATION_GAP_MIN)
        & F.col(f"endpoint.{state}").isNotNull()
        & (
            F.col("label_end_microseconds") - F.col("last_future_poll")
            <= maximum_gap
        )
    )

    features = features.withColumn(
        f"{outcome}_target",
        F.when(
            (F.col(state) == False) & covered,
            F.max(F.col(state).cast("double")).over(future_window),
        ),
    )

# Keep the coverage endpoint for chronological train/validation splitting.
features = features.drop(
    "future_poll_count",
    "last_future_poll",
    "largest_future_gap",
    "endpoint",
)

display(features.agg(
    F.count("bike_target").alias("bike_labeled_polls"),
    F.avg("bike_target").alias("bike_onset_share"),
    F.count("dock_target").alias("dock_labeled_polls"),
    F.avg("dock_target").alias("dock_onset_share"),
))

## Write the Research Gold Table

I am hoping this helps speed up execution. Serverless doesn't allow caching of results so there is a lot of duplicate
data processing that occurs

In [0]:
features.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("citibike_project.citibike.research_station_features")

features = spark.table(
    "citibike_project.citibike.research_station_features"
)

# Validation Set

We will construct a one week validation set (that is all the time we have at the moment)



In [0]:
VALIDATION_START = "2026-09-21T04:00:00Z"

split_time = F.expr(
    f"unix_micros(to_timestamp('{VALIDATION_START}'))"
)

training = features.filter(
    F.col("label_end_microseconds") < split_time
)

validation = features.filter(
    F.col("poll_microseconds") >= split_time
)

display(
    training.select(
        F.lit("Training").alias("split"),
        "fetched_at", "bike_target", "dock_target",
    )
    .unionByName(
        validation.select(
            F.lit("Validation").alias("split"),
            "fetched_at", "bike_target", "dock_target",
        )
    )
    .groupBy("split")
    .agg(
        F.min("fetched_at").alias("first_poll"),
        F.max("fetched_at").alias("last_poll"),
        F.count("bike_target").alias("bike_examples"),
        F.avg("bike_target").alias("bike_onset_share"),
        F.count("dock_target").alias("dock_examples"),
        F.avg("dock_target").alias("dock_onset_share"),
    )
)

## Logistic Regression Model training and testing

In [0]:


predictor_columns = [
    "available",
    "available_fraction",
    "change_30min_fraction",
    "log_capacity",
    "weekend",
    "hour_sin_1", "hour_cos_1",
    "hour_sin_2", "hour_cos_2",
]

def model_data(frame, outcome):
    return (
        frame
        .filter(F.col(f"{outcome}_target").isNotNull())
        .select(
            "station_id", "lat", "lon", "hour", "geographic_cell",
            "log_capacity", "weekend",
            "hour_sin_1", "hour_cos_1",
            "hour_sin_2", "hour_cos_2",
            F.col(f"{outcome}_target").alias("label"),
            F.col(f"num_{outcome}s_available").cast("double").alias("available"),
            F.col(f"{outcome}_available_fraction").alias("available_fraction"),
            F.col(f"{outcome}_change_30min_per_minute_fraction")
                .alias("change_30min_fraction"),
        )
    )


models = {}
predictions = {}
score_rows = []

for outcome in ["bike", "dock"]:
    train = model_data(training, outcome)
    valid = model_data(validation, outcome)

    # Verify that both datasets contain positives and negatives.
    for name, frame in [("Training", train), ("Validation", valid)]:
        if frame.select("label").distinct().count() != 2:
            raise ValueError(f"{outcome}: {name} needs both label classes.")

    imputed_columns = [f"{c}_imputed" for c in predictor_columns]

    pipeline = Pipeline(stages=[
        Imputer(
            inputCols=predictor_columns,
            outputCols=imputed_columns,
            strategy="median",
        ),
        VectorAssembler(
            inputCols=imputed_columns,
            outputCol="model_features",
        ),
        LogisticRegression(
            featuresCol="model_features",
            labelCol="label",
            regParam=0.0,
            maxIter=100,
            standardization=True,
        ),
    ])

    model = pipeline.fit(train)
    models[outcome] = model

    result = (
        model.transform(valid)
        .withColumn("probability_value", vector_to_array("probability")[1])
        .select(
            "station_id", "lat", "lon", "hour", "label", "probability_value"
        )
        .withColumn("residual", F.col("label") - F.col("probability_value"))
    )

    # Clip only for numerical stability when taking logarithms.
    probability = F.greatest(
        F.lit(1e-12),
        F.least(F.lit(1 - 1e-12), F.col("probability_value")),
    )

    result = result.withColumn(
        "log_loss",
        -F.col("label") * F.log(probability)
        -(1 - F.col("label")) * F.log(1 - probability),
    )

    predictions[outcome] = result

    scores = result.agg(
        F.count("*").alias("validation_examples"),
        F.avg("label").alias("event_rate"),
        F.avg("probability_value").alias("mean_prediction"),
        F.avg("log_loss").alias("log_loss"),
    ).first().asDict()

    scores["roc_auc"] = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="probability_value",
        metricName="areaUnderROC",
    ).evaluate(result)

    # Constant-probability benchmark: training event frequency.
    prevalence = train.agg(F.avg("label")).first()[0]
    event_rate = scores["event_rate"]

    scores["constant_log_loss"] = float(
        -event_rate * np.log(prevalence)
        -(1 - event_rate) * np.log(1 - prevalence)
    )
    scores["outcome"] = outcome
    score_rows.append(scores)

    if model.stages[-1].summary.totalIterations >= 100:
        print(f"{outcome}: reached iteration limit; check convergence.")

display(spark.createDataFrame(score_rows))


## Logistic Model Error Exploration

We are going to split this into two cells because the computations take a long time. The first cell will
compute everything needed for the figure. The second cell will plot.
This allows figures to be improved 

In [0]:
station_diagnostics = {}
calibration_tables = {}

fig, axes = plt.subplots(
    2, 3, figsize=(17, 10), constrained_layout=True
)

for row, outcome in enumerate(["bike", "dock"]):
    result = predictions[outcome]

    # Calibration: observed frequency versus predicted probability.
    calibration = (
        result
        .withColumn(
            "probability_bin",
            F.least(F.floor(F.col("probability_value") * 20), F.lit(19)),
        )
        .groupBy("probability_bin")
        .agg(
            F.count("*").alias("polls"),
            F.avg("probability_value").alias("predicted"),
            F.avg("label").alias("observed"),
        )
        .orderBy("probability_bin")
        .toPandas()
    )
    calibration_tables[outcome] = calibration

    ax = axes[row, 0]
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.plot(calibration.predicted, calibration.observed, "o-")
    ax.set(
        title=f"{outcome.title()} · calibration",
        xlabel="Mean predicted probability",
        ylabel="Observed onset fraction",
        xlim=(0, 1), ylim=(0, 1),
    )

    # ROC: cumulative positives and negatives from highest score down.
    roc = (
        result
        .withColumn(
            "score_bin",
            F.least(F.floor(F.col("probability_value") * 1000), F.lit(999)),
        )
        .groupBy("score_bin")
        .agg(
            F.sum("label").alias("positives"),
            F.sum(1 - F.col("label")).alias("negatives"),
        )
        .orderBy(F.desc("score_bin"))
        .toPandas()
    )

    true_positive_rate = np.r_[
        0, roc.positives.cumsum() / roc.positives.sum()
    ]
    false_positive_rate = np.r_[
        0, roc.negatives.cumsum() / roc.negatives.sum()
    ]

    ax = axes[row, 1]
    ax.plot(false_positive_rate, true_positive_rate)
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set(
        title=f"{outcome.title()} · ROC",
        xlabel="False-positive rate",
        ylabel="True-positive rate",
        xlim=(0, 1), ylim=(0, 1),
    )

    hourly_error = (
        result.groupBy("hour")
        .agg(F.avg("residual").alias("mean_residual"))
        .orderBy("hour")
        .toPandas()
    )

    ax = axes[row, 2]
    ax.plot(hourly_error.hour, 100 * hourly_error.mean_residual, "o-")
    ax.axhline(0, color="gray", linestyle="--")
    ax.set(
        title=f"{outcome.title()} · time residual",
        xlabel="New York hour",
        ylabel="Observed minus predicted (percentage points)",
    )

    # Location is included in the grouping to preserve station relocations.
    station = (
        result.groupBy("station_id", "lat", "lon")
        .agg(
            F.count("*").alias("polls"),
            F.avg("label").alias("observed"),
            F.avg("probability_value").alias("predicted"),
            F.avg("residual").alias("mean_residual"),
            F.avg("log_loss").alias("mean_log_loss"),
            F.sum("log_loss").alias("total_log_loss"),
        )
        .toPandas()
    )
    station["loss_share_percent"] = (
        100 * station.total_log_loss / station.total_log_loss.sum()
    )
    station_diagnostics[outcome] = station

fig.suptitle("Baseline logistic regression · development validation", fontsize=22)
plt.show()

# Geographic diagnostics.
fig, axes = plt.subplots(
    2, 2, figsize=(13, 14), constrained_layout=True
)
residual_scale = TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)

for row, outcome in enumerate(["bike", "dock"]):
    station = station_diagnostics[outcome]
    q = station.dropna(subset=["lat", "lon"])

    # Cap the displayed scale at the 95th percentile, not the data itself.
    loss_limit = max(float(station.mean_log_loss.quantile(.95)), 1e-6)
    loss_scale = Normalize(0, loss_limit)

    for ax in axes[row]:
        map_background(ax)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticks([])
        ax.set_yticks([])

    axes[row, 0].scatter(
        q.lon, q.lat, c=q.mean_log_loss,
        cmap="viridis", norm=loss_scale, s=10, zorder=2,
    )
    axes[row, 0].set_title(f"{outcome.title()} · mean log loss")
    fig.colorbar(
        ScalarMappable(norm=loss_scale, cmap="viridis"),
        ax=axes[row, 0], shrink=.6, extend="max",
        label="Mean log loss",
    )

    axes[row, 1].scatter(
        q.lon, q.lat, c=100 * q.mean_residual,
        cmap="RdBu_r", norm=residual_scale, s=10, zorder=2,
    )
    axes[row, 1].set_title(f"{outcome.title()} · signed error")

fig.colorbar(
    ScalarMappable(norm=residual_scale, cmap="RdBu_r"),
    ax=axes[:, 1].tolist(), shrink=.6, extend="both",
    label="Observed minus predicted (percentage points)",
)
fig.suptitle(
    "Where does the baseline struggle?\nRed residuals indicate underprediction",
    fontsize=22,
)
plt.show()

for outcome in ["bike", "dock"]:
    print(f"{outcome.title()}: largest contributions to validation log loss")
    display(
        station_diagnostics[outcome]
        .sort_values("total_log_loss", ascending=False)
        .head(10)
    )

In [0]:
calibration_tables = {}
roc_tables = {}
hourly_residual_tables = {}
station_diagnostics = {}

for outcome in ["bike", "dock"]:
    result = predictions[outcome]

    calibration_tables[outcome] = (
        result
        .withColumn(
            "probability_bin",
            F.least(F.floor(F.col("probability_value") * 20), F.lit(19)),
        )
        .groupBy("probability_bin")
        .agg(
            F.count("*").alias("polls"),
            F.avg("probability_value").alias("predicted"),
            F.avg("label").alias("observed"),
        )
        .orderBy("probability_bin")
        .toPandas()
    )

    roc = (
        result
        .withColumn(
            "score_bin",
            F.least(F.floor(F.col("probability_value") * 1000), F.lit(999)),
        )
        .groupBy("score_bin")
        .agg(
            F.sum("label").alias("positives"),
            F.sum(1 - F.col("label")).alias("negatives"),
        )
        .orderBy(F.desc("score_bin"))
        .toPandas()
    )
    roc["true_positive_rate"] = (
        roc.positives.cumsum() / roc.positives.sum()
    )
    roc["false_positive_rate"] = (
        roc.negatives.cumsum() / roc.negatives.sum()
    )
    roc_tables[outcome] = roc

    hourly_residual_tables[outcome] = (
        result.groupBy("hour")
        .agg(
            F.count("*").alias("polls"),
            F.avg("label").alias("observed"),
            F.avg("probability_value").alias("predicted"),
            F.avg("residual").alias("mean_residual"),
        )
        .orderBy("hour")
        .toPandas()
    )

    station = (
        result.groupBy("station_id", "lat", "lon")
        .agg(
            F.count("*").alias("polls"),
            F.avg("label").alias("observed"),
            F.avg("probability_value").alias("predicted"),
            F.avg("residual").alias("mean_residual"),
            F.avg("log_loss").alias("mean_log_loss"),
            F.sum("log_loss").alias("total_log_loss"),
        )
        .toPandas()
    )
    station["loss_share_percent"] = (
        100 * station.total_log_loss / station.total_log_loss.sum()
    )
    station_diagnostics[outcome] = station

In [0]:
fig, axes = plt.subplots(
    2, 3, figsize=(17, 10), constrained_layout=True
)

for row, outcome in enumerate(["bike", "dock"]):
    calibration = calibration_tables[outcome]
    roc = roc_tables[outcome]
    hourly = hourly_residual_tables[outcome]

    ax = axes[row, 0]
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.plot(calibration.predicted, calibration.observed, "o-")
    ax.set(
        title=f"{outcome.title()} · calibration",
        xlabel="Mean predicted probability",
        ylabel="Observed onset fraction",
        xlim=(0, 1), ylim=(0, 1),
    )

    ax = axes[row, 1]
    ax.plot(
        np.r_[0, roc.false_positive_rate],
        np.r_[0, roc.true_positive_rate],
    )
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set(
        title=f"{outcome.title()} · ROC",
        xlabel="False-positive rate",
        ylabel="True-positive rate",
        xlim=(0, 1), ylim=(0, 1),
    )

    ax = axes[row, 2]
    ax.plot(hourly.hour, 100 * hourly.mean_residual, "o-")
    ax.axhline(0, color="gray", linestyle="--")
    ax.set(
        title=f"{outcome.title()} · time residual",
        xlabel="New York hour",
        ylabel="Observed minus predicted (percentage points)",
    )

for ax in axes.flat:
    ax.title.set_fontsize(16)
    ax.xaxis.label.set_size(13)
    ax.yaxis.label.set_size(13)
    ax.tick_params(labelsize=11)

fig.suptitle(
    "Baseline logistic regression development validation",
    fontsize=24,
    fontweight="bold",
    x=0.05,
    y=0.99,
    ha="left",
)

fig.text(
    0.05, 0.94,
    textwrap.fill("""Notable residual pattern in prodbability remains, peaking just prior to the morning commute. Model calibration curves suggest 
                  very few data points at the upper end of predicted probability.""",width=140),
    fontsize=14,
    ha="left",
    va="top",
    linespacing=1.4,
)

fig.get_layout_engine().set(rect=(0, 0, 1, 0.84))

plt.show()


In [0]:
fig, axes = plt.subplots(
    2, 2, figsize=(13, 14), constrained_layout=True
)

residual_scale = TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)

for row, outcome in enumerate(["bike", "dock"]):
    station = station_diagnostics[outcome]
    q = station.dropna(subset=["lat", "lon"])

    loss_limit = max(float(station.mean_log_loss.quantile(.95)), 1e-6)
    loss_scale = Normalize(0, loss_limit)

    for ax in axes[row]:
        map_background(ax)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticks([])
        ax.set_yticks([])

    axes[row, 0].scatter(
        q.lon, q.lat,
        c=q.mean_log_loss,
        cmap="viridis", norm=loss_scale,
        s=10, zorder=2,
    )
    axes[row, 0].set_title(
        f"{outcome.title()} · mean log loss",
        fontsize=18,
    )

    colorbar = fig.colorbar(
        ScalarMappable(norm=loss_scale, cmap="viridis"),
        ax=axes[row, 0],
        shrink=.6,
        extend="max",
    )
    colorbar.set_label("Mean log loss", fontsize=14)
    colorbar.ax.tick_params(labelsize=12)

    axes[row, 1].scatter(
        q.lon, q.lat,
        c=100 * q.mean_residual,
        cmap="RdBu_r", norm=residual_scale,
        s=10, zorder=2,
    )
    axes[row, 1].set_title(
        f"{outcome.title()} · signed error",
        fontsize=18,
    )

colorbar = fig.colorbar(
    ScalarMappable(norm=residual_scale, cmap="RdBu_r"),
    ax=axes[:, 1].tolist(),
    shrink=.6,
    extend="both",
)
colorbar.set_label(
    "Observed minus predicted (percentage points)",
    fontsize=14,
)
colorbar.ax.tick_params(labelsize=12)

fig.suptitle(
    "Geographic errors in the baseline model",
    fontsize=24,
    fontweight="bold",
)



annotation = """The primary concentration to prediction errors for bikes are Harlem/Upper Manhattan, Midtown, 
                  and Prospect Park. The major contribution to prediction error for full docks is lower Manhattan and the Upper East Side.
                  In both cases the residual plot indicates that the worst predicted regions under-predict the probability of events."""

fig.text(
    0.05, 0.94,
    textwrap.fill(" ".join(annotation.split()), width=140),
    fontsize=14,
    ha="left", va="top",
    linespacing=1.4,
)


fig.get_layout_engine().set(rect=(0, 0, 1, 0.88))
plt.show()

# Model Variant Two: More geography and time

We build another logistic regression model this time incorporating interactions between geographic boxes and our time features

In [0]:


REGULARIZATION = 0.0 # We have added a bit of features but still leaving this

daily_harmonics = [
    "hour_sin_1", "hour_cos_1",
    "hour_sin_2", "hour_cos_2",
]

geographic_models = {}
geographic_predictions = {}

def prediction_frame(model, data):
    return (
        model.transform(data)
        .withColumn("probability_value", vector_to_array("probability")[1])
        .select(
            "station_id", "lat", "lon", "hour",
            "label", "probability_value",
        )
    )

for outcome in ["bike", "dock"]:
    train = model_data(training, outcome)
    valid = model_data(validation, outcome)

    # Learn geographic categories from training only.
    indexer = StringIndexer(
        inputCol="geographic_cell",
        outputCol="geographic_index",
        handleInvalid="keep",
    ).fit(train)

    number_of_cells = len(indexer.labels)
    if number_of_cells < 2:
        raise ValueError(f"{outcome}: need at least two geographic cells.")

    print(
        outcome,
        "geographic cells:", number_of_cells,
        "reference cell:", indexer.labels[0],
    )

    imputed_columns = [f"{c}_imputed" for c in predictor_columns]

    pipeline = Pipeline(stages=[
        Imputer(
            inputCols=predictor_columns,
            outputCols=imputed_columns,
            strategy="median",
        ),

        indexer,

        OneHotEncoder(
            inputCols=["geographic_index"],
            outputCols=["all_geographic_indicators"],
            dropLast=False,
        ),

        # Exclude the reference cell and the unknown-category indicator.
        VectorSlicer(
            inputCol="all_geographic_indicators",
            outputCol="geographic_indicators",
            indices=list(range(1, number_of_cells)),
        ),

        VectorAssembler(
            inputCols=[f"{c}_imputed" for c in daily_harmonics],
            outputCol="daily_curve",
        ),

        Interaction(
            inputCols=["geographic_indicators", "daily_curve"],
            outputCol="geographic_daily_curves",
        ),

        VectorAssembler(
            inputCols=imputed_columns + [
                "geographic_indicators",
                "geographic_daily_curves",
            ],
            outputCol="model_features",
        ),

        LogisticRegression(
            featuresCol="model_features",
            labelCol="label",
            regParam=REGULARIZATION,
            elasticNetParam=0.0,
            standardization=True,
            maxIter=200,
        ),
    ])

    model = pipeline.fit(train)
    geographic_models[outcome] = model
    geographic_predictions[outcome] = prediction_frame(model, valid)

    print(
        outcome,
        "iterations:", model.stages[-1].summary.totalIterations,
    )

    if model.stages[-1].summary.totalIterations >= 200:
        print("Iteration limit reached: check convergence before interpreting.")

There is a convergence issue for this model. I investigated it and it didn't seem to be critical, it is more just that some errors remain even after a large number of iterations. I did not choose to add regularization. 

In [0]:
# We are making some explicit functions to hopefully save some code


def evaluate_predictions(prediction_data):
    probability = F.greatest(
        F.lit(1e-12),
        F.least(F.lit(1 - 1e-12), F.col("probability_value")),
    )

    result = (
        prediction_data
        .withColumn(
            "residual",
            F.col("label") - F.col("probability_value"),
        )
        .withColumn(
            "log_loss",
            -F.col("label") * F.log(probability)
            -(1 - F.col("label")) * F.log(1 - probability),
        )
    )

    scores = result.agg(
        F.count("*").alias("validation_examples"),
        F.avg("label").alias("event_rate"),
        F.avg("probability_value").alias("mean_prediction"),
        F.avg("log_loss").alias("log_loss"),
    ).first().asDict()

    scores["roc_auc"] = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="probability_value",
        metricName="areaUnderROC",
    ).evaluate(result)

    calibration = (
        result
        .withColumn(
            "probability_bin",
            F.least(F.floor(F.col("probability_value") * 20), F.lit(19)),
        )
        .groupBy("probability_bin")
        .agg(
            F.count("*").alias("polls"),
            F.avg("probability_value").alias("predicted"),
            F.avg("label").alias("observed"),
        )
        .orderBy("probability_bin")
        .toPandas()
    )

    roc = (
        result
        .withColumn(
            "score_bin",
            F.least(F.floor(F.col("probability_value") * 1000), F.lit(999)),
        )
        .groupBy("score_bin")
        .agg(
            F.sum("label").alias("positives"),
            F.sum(1 - F.col("label")).alias("negatives"),
        )
        .orderBy(F.desc("score_bin"))
        .toPandas()
    )
    roc["true_positive_rate"] = (
        roc.positives.cumsum() / roc.positives.sum()
    )
    roc["false_positive_rate"] = (
        roc.negatives.cumsum() / roc.negatives.sum()
    )

    hourly = (
        result.groupBy("hour")
        .agg(
            F.count("*").alias("polls"),
            F.avg("label").alias("observed"),
            F.avg("probability_value").alias("predicted"),
            F.avg("residual").alias("mean_residual"),
        )
        .orderBy("hour")
        .toPandas()
    )

    station = (
        result.groupBy("station_id", "lat", "lon")
        .agg(
            F.count("*").alias("polls"),
            F.avg("label").alias("observed"),
            F.avg("probability_value").alias("predicted"),
            F.avg("residual").alias("mean_residual"),
            F.avg("log_loss").alias("mean_log_loss"),
            F.sum("log_loss").alias("total_log_loss"),
        )
        .toPandas()
    )
    station["loss_share_percent"] = (
        100 * station.total_log_loss / station.total_log_loss.sum()
    )

    return {
        "scores": scores,
        "calibration": calibration,
        "roc": roc,
        "hourly": hourly,
        "station": station,
    }


geographic_diagnostics = {}
geographic_score_rows = []

for outcome in ["bike", "dock"]:
    diagnostics = evaluate_predictions(geographic_predictions[outcome])
    geographic_diagnostics[outcome] = diagnostics

    geographic_score_rows.append({
        "outcome": outcome,
        "model": "Geography × daily harmonics",
        **diagnostics["scores"],
    })

display(spark.createDataFrame(geographic_score_rows))

The performance of the geographic-time model improves noticeably above the baseline logistic regression model.

In [0]:


fig, axes = plt.subplots(
    2, 3, figsize=(17, 10), constrained_layout=True
)

for row, outcome in enumerate(["bike", "dock"]):
    diagnostics = geographic_diagnostics[outcome]
    calibration = diagnostics["calibration"]
    roc = diagnostics["roc"]
    hourly = diagnostics["hourly"]

    ax = axes[row, 0]
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.plot(calibration.predicted, calibration.observed, "o-")
    ax.set(
        title=f"{outcome.title()}: calibration",
        xlabel="Mean predicted probability",
        ylabel="Observed onset fraction",
        xlim=(0, 1), ylim=(0, 1),
    )

    ax = axes[row, 1]
    ax.plot(
        np.r_[0, roc.false_positive_rate],
        np.r_[0, roc.true_positive_rate],
    )
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set(
        title=f"{outcome.title()} · ROC",
        xlabel="False-positive rate",
        ylabel="True-positive rate",
        xlim=(0, 1), ylim=(0, 1),
    )

    ax = axes[row, 2]
    ax.plot(hourly.hour, 100 * hourly.mean_residual, "o-")
    ax.axhline(0, color="gray", linestyle="--")
    ax.set(
        title=f"{outcome.title()}: time residual",
        xlabel="New York hour",
        ylabel="Observed minus predicted (percentage points)",
    )

for ax in axes.flat:
    ax.title.set_fontsize(16)
    ax.xaxis.label.set_size(13)
    ax.yaxis.label.set_size(13)
    ax.tick_params(labelsize=11)

fig.suptitle(
    "Geographic logistic regression: development validation",
    fontsize=24,
    fontweight="bold",
    x=0.05, y=0.99, ha="left",
)

annotation = """This model improves upon the overal performance of the baseline model, but the residual patterns are quite similar. Calibration
is excellent and even maintains at the high end where there are relatively few examples."""

fig.text(
    0.05, 0.94,
    textwrap.fill(" ".join(annotation.split()), width=140),
    fontsize=14,
    ha="left", va="top",
    linespacing=1.4,
)

fig.get_layout_engine().set(rect=(0, 0, 1, 0.84))
plt.show()

In [0]:
fig, axes = plt.subplots(
    2, 2, figsize=(13, 14), constrained_layout=True
)

residual_scale = TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)

for row, outcome in enumerate(["bike", "dock"]):
    station = geographic_diagnostics[outcome]["station"]
    q = station.dropna(subset=["lat", "lon"])

    # Same scale as the earlier baseline map.
    loss_limit = max(
        float(station_diagnostics[outcome].mean_log_loss.quantile(.95)),
        1e-6,
    )
    loss_scale = Normalize(0, loss_limit)

    for ax in axes[row]:
        map_background(ax)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticks([])
        ax.set_yticks([])

    axes[row, 0].scatter(
        q.lon, q.lat,
        c=q.mean_log_loss,
        cmap="viridis", norm=loss_scale,
        s=10, zorder=2,
    )
    axes[row, 0].set_title(
        f"{outcome.title()}: mean log loss",
        fontsize=18,
    )

    colorbar = fig.colorbar(
        ScalarMappable(norm=loss_scale, cmap="viridis"),
        ax=axes[row, 0],
        shrink=.6, extend="max",
    )
    colorbar.set_label("Mean log loss", fontsize=14)
    colorbar.ax.tick_params(labelsize=12)

    axes[row, 1].scatter(
        q.lon, q.lat,
        c=100 * q.mean_residual,
        cmap="RdBu_r", norm=residual_scale,
        s=10, zorder=2,
    )
    axes[row, 1].set_title(
        f"{outcome.title()}: signed error",
        fontsize=18,
    )

colorbar = fig.colorbar(
    ScalarMappable(norm=residual_scale, cmap="RdBu_r"),
    ax=axes[:, 1].tolist(),
    shrink=.6, extend="both",
)
colorbar.set_label(
    "Observed minus predicted (percentage points)",
    fontsize=14,
)
colorbar.ax.tick_params(labelsize=12)

fig.suptitle(
    "Geographic errors after adding local daily patterns",
    fontsize=24,
    fontweight="bold",
    x=0.05, y=0.99, ha="left",
)

annotation = """Despite the addition of geographic features, the same residual pattern persists"""

fig.text(
    0.05, 0.94,
    textwrap.fill(" ".join(annotation.split()), width=110),
    fontsize=14,
    ha="left", va="top",
    linespacing=1.4,
)

fig.get_layout_engine().set(rect=(0, 0, 1, 0.84))
plt.show()

In [0]:


NUM_TREES = 50
MAX_DEPTH = 8
MAX_BINS = 64

forest_predictors = predictor_columns + ["lat", "lon"]

forest_models = {}
forest_predictions = {}

for outcome in ["bike", "dock"]:
    train = model_data(training, outcome)
    valid = model_data(validation, outcome)

    imputed_columns = [f"{c}_imputed" for c in forest_predictors]

    pipeline = Pipeline(stages=[
        Imputer(
            inputCols=forest_predictors,
            outputCols=imputed_columns,
            strategy="median",
        ),

        VectorAssembler(
            inputCols=imputed_columns,
            outputCol="model_features",
        ),

        RandomForestClassifier(
            featuresCol="model_features",
            labelCol="label",
            numTrees=NUM_TREES,
            maxDepth=MAX_DEPTH,
            maxBins=MAX_BINS,
            minInstancesPerNode=100,
            featureSubsetStrategy="sqrt",
            subsamplingRate=0.5,
            bootstrap=True,
            seed=173,
        ),
    ])

    started = perf_counter()

    model = pipeline.fit(train)
    forest_models[outcome] = model

    print(
        f"{outcome}: pipeline fitting took "
        f"{(perf_counter() - started) / 60:.1f} minutes"
    )

    forest_predictions[outcome] = prediction_frame(model, valid)

In [0]:
forest_diagnostics = {}
forest_score_rows = []

for outcome in ["bike", "dock"]:
    diagnostics = evaluate_predictions(forest_predictions[outcome])
    forest_diagnostics[outcome] = diagnostics

    forest_score_rows.append({
        "outcome": outcome,
        "model": "Random forest",
        **diagnostics["scores"],
    })

display(spark.createDataFrame(forest_score_rows))

The Random Forest model surprised me by performing more poorly than either logistic regression model both in terms of relative ranking and calibration. I have a hunch I have not done things correctly, but am not going to do a deep dive right now.

In [0]:


fig, axes = plt.subplots(
    2, 3, figsize=(17, 10), constrained_layout=True
)

for row, outcome in enumerate(["bike", "dock"]):
    diagnostics = forest_diagnostics[outcome]
    calibration = diagnostics["calibration"]
    roc = diagnostics["roc"]
    hourly = diagnostics["hourly"]

    ax = axes[row, 0]
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.plot(calibration.predicted, calibration.observed, "o-")
    ax.set(
        title=f"{outcome.title()}: calibration",
        xlabel="Mean predicted probability",
        ylabel="Observed onset fraction",
        xlim=(0, 1), ylim=(0, 1),
    )

    ax = axes[row, 1]
    ax.plot(
        np.r_[0, roc.false_positive_rate],
        np.r_[0, roc.true_positive_rate],
    )
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set(
        title=f"{outcome.title()}: ROC",
        xlabel="False-positive rate",
        ylabel="True-positive rate",
        xlim=(0, 1), ylim=(0, 1),
    )

    ax = axes[row, 2]
    ax.plot(hourly.hour, 100 * hourly.mean_residual, "o-")
    ax.axhline(0, color="gray", linestyle="--")
    ax.set(
        title=f"{outcome.title()}: time residual",
        xlabel="New York hour",
        ylabel="Observed minus predicted (percentage points)",
    )

for ax in axes.flat:
    ax.title.set_fontsize(16)
    ax.xaxis.label.set_size(13)
    ax.yaxis.label.set_size(13)
    ax.tick_params(labelsize=11)

fig.suptitle(
    "Geographic logistic regression: development validation",
    fontsize=24,
    fontweight="bold",
    x=0.05, y=0.99, ha="left",
)

annotation = """The temporal residuals show a difference to the two logistic regression models. Calibration is naturally poorer, underpredicting
probabilities of stockouts when they are more likely. The ROC curves should be convex, but one has a slight divot."""

fig.text(
    0.05, 0.94,
    textwrap.fill(" ".join(annotation.split()), width=140),
    fontsize=14,
    ha="left", va="top",
    linespacing=1.4,
)

fig.get_layout_engine().set(rect=(0, 0, 1, 0.84))
plt.show()

In [0]:
fig, axes = plt.subplots(
    2, 2, figsize=(13, 14), constrained_layout=True
)

residual_scale = TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)

for row, outcome in enumerate(["bike", "dock"]):
    station = forest_diagnostics[outcome]["station"]
    q = station.dropna(subset=["lat", "lon"])

    # Same scale as the earlier baseline map.
    loss_limit = max(
        float(station_diagnostics[outcome].mean_log_loss.quantile(.95)),
        1e-6,
    )
    loss_scale = Normalize(0, loss_limit)

    for ax in axes[row]:
        map_background(ax)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticks([])
        ax.set_yticks([])

    axes[row, 0].scatter(
        q.lon, q.lat,
        c=q.mean_log_loss,
        cmap="viridis", norm=loss_scale,
        s=10, zorder=2,
    )
    axes[row, 0].set_title(
        f"{outcome.title()}: mean log loss",
        fontsize=18,
    )

    colorbar = fig.colorbar(
        ScalarMappable(norm=loss_scale, cmap="viridis"),
        ax=axes[row, 0],
        shrink=.6, extend="max",
    )
    colorbar.set_label("Mean log loss", fontsize=14)
    colorbar.ax.tick_params(labelsize=12)

    axes[row, 1].scatter(
        q.lon, q.lat,
        c=100 * q.mean_residual,
        cmap="RdBu_r", norm=residual_scale,
        s=10, zorder=2,
    )
    axes[row, 1].set_title(
        f"{outcome.title()} · signed error",
        fontsize=18,
    )

colorbar = fig.colorbar(
    ScalarMappable(norm=residual_scale, cmap="RdBu_r"),
    ax=axes[:, 1].tolist(),
    shrink=.6, extend="both",
)
colorbar.set_label(
    "Observed minus predicted (percentage points)",
    fontsize=14,
)
colorbar.ax.tick_params(labelsize=12)

fig.suptitle(
    "Geographic errors: Random Forest Model",
    fontsize=24,
    fontweight="bold",
    x=0.05, y=0.99, ha="left",
)

annotation = """The Random Forest model shares the same pattern of geographic residuals as the other two models.
This surprises me given that it is the most flexible model. I will try again with different settings and then just leave it."""

fig.text(
    0.05, 0.94,
    textwrap.fill(" ".join(annotation.split()), width=110),
    fontsize=14,
    ha="left", va="top",
    linespacing=1.4,
)

fig.get_layout_engine().set(rect=(0, 0, 1, 0.84))
plt.show()

In [0]:
import pandas as pd

# Compare validation performance.
comparison = pd.concat(
    [
        pd.DataFrame(score_rows).assign(model="Baseline LR"),
        pd.DataFrame(geographic_score_rows).assign(model="Geographic LR"),
        pd.DataFrame(forest_score_rows).assign(model="Random forest"),
    ],
    ignore_index=True,
)

display(
    comparison[
        [
            "outcome", "model", "validation_examples",
            "event_rate", "log_loss", "roc_auc",
        ]
    ].sort_values(["outcome", "log_loss"])
)

# Overlay the precomputed ROC curves.
fig, axes = plt.subplots(
    1, 2, figsize=(12, 5), constrained_layout=True
)

for ax, outcome in zip(axes, ["bike", "dock"]):
    curves = [
        ("Baseline LR", roc_tables[outcome]),
        ("Geographic LR", geographic_diagnostics[outcome]["roc"]),
        ("Random forest", forest_diagnostics[outcome]["roc"]),
    ]

    for name, curve in curves:
        auc = comparison.loc[
            (comparison.outcome == outcome)
            & (comparison.model == name),
            "roc_auc",
        ].iloc[0]

        ax.plot(
            np.r_[0, curve.false_positive_rate],
            np.r_[0, curve.true_positive_rate],
            label=f"{name} · AUC {auc:.4f}",
            linewidth=2,
        )

    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set(
        title=f"{outcome.title()} stockout onset",
        xlabel="False-positive rate",
        ylabel="True-positive rate",
        xlim=(0, 1),
        ylim=(0, 1),
    )
    ax.legend(fontsize=10)
    ax.title.set_fontsize(16)

fig.suptitle(
    "Model comparison: development validation",
    fontsize=22,
    fontweight="bold",
)
plt.show()

In [0]:


forest2_models = {}
forest2_predictions = {}

forest2_predictors = predictor_columns + ["lat", "lon"]

for outcome in ["bike", "dock"]:
    train = model_data(training, outcome)
    valid = model_data(validation, outcome)

    imputed_columns = [f"{c}_imputed" for c in forest2_predictors]

    pipeline = Pipeline(stages=[
        Imputer(
            inputCols=forest2_predictors,
            outputCols=imputed_columns,
            strategy="median",
        ),
        VectorAssembler(
            inputCols=imputed_columns,
            outputCol="model_features",
        ),
        RandomForestClassifier(
            featuresCol="model_features",
            labelCol="label",
            numTrees=50,
            maxDepth=12,
            maxBins=64,
            minInstancesPerNode=100,
            featureSubsetStrategy="all",
            subsamplingRate=0.5,
            bootstrap=True,
            seed=173,
        ),
    ])

    started = perf_counter()
    model = pipeline.fit(train)

    forest2_models[outcome] = model
    forest2_predictions[outcome] = prediction_frame(model, valid)

    print(
        f"{outcome}: fitting took "
        f"{(perf_counter() - started) / 60:.1f} minutes"
    )

In [0]:
forest2_diagnostics = {}
forest2_score_rows = []

for outcome in ["bike", "dock"]:
    diagnostics = evaluate_predictions(forest2_predictions[outcome])
    forest2_diagnostics[outcome] = diagnostics

    forest2_score_rows.append({
        "outcome": outcome,
        "model": "Random forest 2",
        **diagnostics["scores"],
    })

comparison = pd.concat(
    [
        pd.DataFrame(score_rows).assign(model="Baseline LR"),
        pd.DataFrame(geographic_score_rows).assign(model="Geographic LR"),
        pd.DataFrame(forest_score_rows).assign(model="Random forest 1"),
        pd.DataFrame(forest2_score_rows),
    ],
    ignore_index=True,
)

display(
    comparison[
        [
            "outcome", "model", "validation_examples",
            "event_rate", "log_loss", "roc_auc",
        ]
    ].sort_values(["outcome", "log_loss"])
)

We eeked a bit more gains out of the Random Forest but nothing that motivates us to go further at the current moment.


## Conclusions

1. There is persistent temporal and geographic heterogeneity in both stockout and full dock risk and in station capacity.
2. There are some regions where stockout risk occurs at all times of the day, though an obvious commutter pattern can be seen throughout Manhattan.
3. Our Georgraphic/Time interaction Logistic Regression model which incorporate (naively constructed) geographic features show improved performance but had identical patterns of residuals.
4. A very flexible random forest model performed quite poorly to my great surprise. 

The goal of this development exercise was to create to a first exemplar to show as a production model, so we will select the geographic logistic regression. I think there is significant room for model improvement and the dataset is quite short. A goal is to develop a hierarchical model which can even better leverage the geographic structure.


## Regularized Model

Some annoyances with

In [0]:
# Model Variant Three: geography × time with L2 shrinkage (partial pooling).
# Each cell's intercept and daily-curve adjustments get a normal(0, tau^2) prior,
# fitted as a MAP estimate via ridge regression with regParam = 1 / (n * tau^2).
# tau is chosen by expanding-window CV inside the training period only,
# so the validation week stays untouched for the model comparison.

from pyspark.ml.feature import StandardScaler

PRIOR_SCALES = [0.001, 0.01, 0.03, 0.1, 0.15, 0.25]   # prior sd of cell effects, log-odds units
TUNING_FOLDS = 2                       # one-week tuning windows before VALIDATION_START
FOLD_DAYS = 7
SHRINKAGE_MAX_ITER = 200

DAY_US = 24 * 60 * 60 * 1_000_000
split_us = int(
    datetime.fromisoformat(VALIDATION_START.replace("Z", "+00:00")).timestamp() * 1_000_000
)

# Tuning windows, oldest first: each trains on everything before its start.
tuning_windows = [
    (split_us - k * FOLD_DAYS * DAY_US, split_us - (k - 1) * FOLD_DAYS * DAY_US)
    for k in range(TUNING_FOLDS, 0, -1)
]


def shrinkage_pipeline(reg_param):
    imputed = [f"{c}_imputed" for c in predictor_columns]
    return Pipeline(stages=[
        Imputer(inputCols=predictor_columns, outputCols=imputed, strategy="median"),
        # Standardize numeric inputs ourselves, so the penalty treats them evenly
        # while cell indicators are penalized in their natural 0/1 units.
        VectorAssembler(inputCols=imputed, outputCol="numeric_raw"),
        StandardScaler(inputCol="numeric_raw", outputCol="numeric_scaled",
                       withMean=True, withStd=True),
        # Every training cell gets an indicator; unseen cells get all zeros.
        StringIndexer(inputCol="geographic_cell", outputCol="geographic_index",
                      handleInvalid="keep"),
        OneHotEncoder(inputCols=["geographic_index"], outputCols=["geographic_indicators"]),
        VectorAssembler(inputCols=[f"{c}_imputed" for c in daily_harmonics],
                        outputCol="daily_curve"),
        Interaction(inputCols=["geographic_indicators", "daily_curve"],
                    outputCol="geographic_daily_curves"),
        VectorAssembler(
            inputCols=["numeric_scaled", "geographic_indicators", "geographic_daily_curves"],
            outputCol="model_features"),
        LogisticRegression(featuresCol="model_features", labelCol="label",
                           regParam=reg_param, elasticNetParam=0.0,
                           standardization=False, maxIter=SHRINKAGE_MAX_ITER),
    ])


def fit_shrinkage(train, prior_scale):
    n = train.count()
    reg_param = 1.0 / (n * prior_scale ** 2)
    model = shrinkage_pipeline(reg_param).fit(train)
    if model.stages[-1].summary.totalIterations >= SHRINKAGE_MAX_ITER:
        print(f"  tau={prior_scale}: iteration limit reached")
    return model, reg_param


def validation_log_loss(model, frame):
    p = F.greatest(F.lit(1e-12),
                   F.least(F.lit(1 - 1e-12), vector_to_array("probability")[1]))
    return model.transform(frame).agg(
        F.count("*").alias("examples"),
        F.avg(-F.col("label") * F.log(p) - (1 - F.col("label")) * F.log(1 - p))
            .alias("log_loss"),
    ).first()


# 1. Tune tau inside the training period.
tuning_rows = []
for outcome in ["bike", "dock"]:
    for fold, (start, end) in enumerate(tuning_windows, 1):
        fold_train = model_data(
            training.filter(F.col("label_end_microseconds") < start), outcome)
        fold_valid = model_data(
            training.filter((F.col("poll_microseconds") >= start)
                            & (F.col("label_end_microseconds") < end)), outcome)
        for prior_scale in PRIOR_SCALES:
            started = perf_counter()
            model, reg_param = fit_shrinkage(fold_train, prior_scale)
            score = validation_log_loss(model, fold_valid)
            tuning_rows.append({
                "outcome": outcome, "fold": fold, "prior_scale": prior_scale,
                "reg_param": reg_param, "examples": score.examples,
                "log_loss": score.log_loss,
            })
            print(f"{outcome} fold {fold} tau={prior_scale}: log loss "
                  f"{score.log_loss:.5f} ({(perf_counter() - started) / 60:.1f} min)")

tuning = pd.DataFrame(tuning_rows)
tuning_summary = (
    tuning.assign(total_loss=tuning.log_loss * tuning.examples)
    .groupby(["outcome", "prior_scale"], as_index=False)[["total_loss", "examples"]].sum()
)
tuning_summary["weighted_log_loss"] = tuning_summary.total_loss / tuning_summary.examples
display(tuning_summary)

best = tuning_summary.loc[tuning_summary.groupby("outcome").weighted_log_loss.idxmin()]
best_prior_scale = dict(zip(best.outcome, best.prior_scale))
print("Chosen prior scale:", best_prior_scale)
for outcome, scale in best_prior_scale.items():
    if scale in (min(PRIOR_SCALES), max(PRIOR_SCALES)):
        print(f"{outcome}: best tau is at the edge of the grid; consider extending it.")

# 2. Refit on all training data and score the untouched validation week.
shrinkage_models, shrinkage_predictions, shrinkage_diagnostics = {}, {}, {}
shrinkage_score_rows = []
for outcome in ["bike", "dock"]:
    train = model_data(training, outcome)
    valid = model_data(validation, outcome)
    model, reg_param = fit_shrinkage(train, best_prior_scale[outcome])
    shrinkage_models[outcome] = model
    shrinkage_predictions[outcome] = prediction_frame(model, valid)
    diagnostics = evaluate_predictions(shrinkage_predictions[outcome])
    shrinkage_diagnostics[outcome] = diagnostics
    shrinkage_score_rows.append({
        "outcome": outcome, "model": "Shrunk geographic LR",
        "prior_scale": best_prior_scale[outcome], "reg_param": reg_param,
        **diagnostics["scores"],
    })

comparison = pd.concat(
    [
        pd.DataFrame(score_rows).assign(model="Baseline LR"),
        pd.DataFrame(geographic_score_rows).assign(model="Geographic LR"),
        pd.DataFrame(forest_score_rows).assign(model="Random forest 1"),
        pd.DataFrame(forest2_score_rows),
        pd.DataFrame(shrinkage_score_rows),
    ],
    ignore_index=True,
)
display(
    comparison[["outcome", "model", "validation_examples", "event_rate",
                "log_loss", "roc_auc"]].sort_values(["outcome", "log_loss"])
)